In [ ]:
from pathlib import Path
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
import sys
import os
sys.path.insert(0, str(Path.cwd().parent / 'direct11'))
sys.path.insert(0, str(Path.cwd().parent / 'models'))
from pathlib import Path
import json
import numpy as np
import pandas as pd
import xarray as xr
try:
    from IPython.display import display
except ImportError:
    display = print

DATA_DIR = RESULTS
MODEL_DIR = setting("YANGTZE_MODEL_DIR", "models")
RAW_DIR = DATA
OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LC, LV, LS = 611.7192928848946, 142.40845542406902, 329.4372079166584
EXPERIMENTS = [
    dict(id='no_pinn', label='No PINN', suffix='', weights=(0.,0.,0.)),
    dict(id='full_pinn', label='Full PINN', suffix='_pinn', weights=(LC,LV,LS)),
    dict(id='no_continuity', label='w/o Continuity', suffix='_pinn_no_continuity', weights=(0.,LV,LS)),
    dict(id='no_vorticity', label='w/o Vorticity', suffix='_pinn_no_vorticity', weights=(LC,0.,LS)),
    dict(id='no_shear', label='w/o Shear', suffix='_pinn_no_shear', weights=(LC,LV,0.)),
]
for exp in EXPERIMENTS:
    exp['model_path'] = str(MODEL_DIR/f'DA_TransUNet_1_24year_changjiang{exp["suffix"]}.pth')

FILE_OVERRIDES = {}  # 示例：{('full_pinn','predicty','u'): Path(r'实际文件.nc')}
TIME_START, TIME_END = None, None
TIME_CHUNK = 48
W_EXTERNAL_SCALE = -0.1
REUSE_CACHE = True
BLOCK_HOURS = (72, 24, 168)  # 第一项是绘图主分析，另两项保存在 NC/CSV。
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 25
FIGURE_DPI = 600

RUN_GRADIENT = True
GRADIENT_CONFIG = dict(
    raw_dir=str(RAW_DIR), device=os.environ.get('YANGTZE_DEVICE', 'cpu'),
    experiment_ids=['full_pinn','no_shear'],
    period_start='2000-01-01T06:00:00', period_end='2023-12-31T23:00:00',
    
    normalization_cache=str(MODEL_DIR/'DA_TransUNet_1_24year_changjiang_normalization.npz'),
    samples_per_season=4,  # 四季各 4 个固定抽样时次，共 16 个单样本批次。
    prediction_rrmse_tolerance=1.e-3,
    truth_rrmse_tolerance=1.e-3,
)

PREFIX = OUTPUT_DIR/'da_transunet_1hour_pinn_physics'
SUMMARY_NC = Path(str(PREFIX)+'_summary.nc')
STATS_NC = Path(str(PREFIX)+'_time_statistics.nc')
MAPS_NC = Path(str(PREFIX)+'_grid_metrics.nc')
BOOTSTRAP_NC = Path(str(PREFIX)+'_block_bootstrap.nc')
GRADIENT_NC = Path(str(PREFIX)+'_parameter_gradients.nc')


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import uuid
import numpy as np
import pandas as pd
import xarray as xr


def open_aligned_results(data_dir, experiments, overrides=None, time_start=None, time_end=None):
    """只读打开全部结果；时间取交集、空间按坐标对齐，不替换任一实验的真实值。"""
    overrides = {} if overrides is None else overrides
    handles, manifest, data = [], [], {}
    common_times, reference = None, None
    paths = []
    for exp in experiments:
        for comp in ['u', 'v', 'w']:
            for kind in ['testy', 'predicty']:
                default = Path(data_dir)/f'da_transunet_{kind}_{comp}_1_24year_changjiang{exp["suffix"]}.nc'
                path = Path(overrides.get((exp['id'], kind, comp), default))
                paths.append((exp['id'], comp, kind, path))
    missing = [str(row[3]) for row in paths if not row[3].is_file()]
    if missing:
        raise FileNotFoundError('以下结果文件不存在，请核对代码框 1 的路径：\n'+'\n'.join(missing))
    try:
        for eid, comp, kind, path in paths:
            ds = xr.open_dataset(path, engine='netcdf4', cache=False, chunks=None)
            handles.append(ds)
            var = kind+'_'+comp
            if var not in ds:
                raise KeyError(f'{path.name} 中没有 {var}，实际变量为 {list(ds.data_vars)}')
            da = ds[var]
            aliases = {'valid_time':'time', 'pressure_level':'level', 'lat':'latitude', 'lon':'longitude'}
            rename = {old:new for old,new in aliases.items() if old in da.dims and new not in da.dims}
            da = da.rename(rename)
            dims = ('time','level','latitude','longitude')
            if set(da.dims) != set(dims):
                raise ValueError(f'{path.name} 需要 time/level/latitude/longitude 四个维度，实际 {da.dims}')
            da = da.transpose(*dims)
            for dim in dims:
                if dim not in da.coords or da[dim].dims != (dim,) or not da.get_index(dim).is_unique:
                    raise ValueError(f'{path.name} 的 {dim} 坐标缺失或存在重复')
            if not np.issubdtype(da.time.dtype, np.datetime64) or np.isnat(da.time.values).any():
                raise ValueError(f'{path.name} 的 time 不能解码为有效 datetime64')
            da = da.sortby('time')
            if time_start is not None or time_end is not None:
                da = da.sel(time=slice(time_start,time_end))
            if da.sizes['time'] < 2:
                raise ValueError(f'{path.name} 选定时次不足 2 个')
            level = np.asarray(da.level.values, dtype=np.float64)
            if np.nanmax(np.abs(level)) > 2000:
                da = da.assign_coords(level=level/100.)
            if reference is None:
                reference = {d:np.asarray(da[d].values).copy() for d in dims[1:]}
                for d in dims[1:]:
                    v = reference[d].astype(float)
                    if not np.isfinite(v).all() or len(v)<2 or not (np.all(np.diff(v)>0) or np.all(np.diff(v)<0)):
                        raise ValueError(f'{d} 必须至少有两个有限、严格单调坐标')
            for dim in dims[1:]:
                target = reference[dim]
                index = da.get_index(dim).get_indexer(target)
                if da.sizes[dim]!=len(target) or (index<0).any():
                    raise ValueError(f'{path.name} 的 {dim} 坐标集合与其他实验不同；不自动插值')
                da = da.isel({dim:index})
            times = da.time.values.astype('datetime64[ns]')
            common_times = times if common_times is None else np.intersect1d(common_times,times)
            data.setdefault(eid,{}).setdefault(comp,{})[kind] = da
            st = path.stat()
            manifest.append(dict(experiment=eid,component=comp,kind=kind,path=str(path.resolve()),
                                 size_bytes=st.st_size,mtime_ns=st.st_mtime_ns,source_count=len(times)))
        if len(common_times)<2:
            raise ValueError('所有实验及变量没有足够的共同测试时次')
        for eid, comp, kind, _ in paths:
            data[eid][comp][kind] = data[eid][comp][kind].sel(time=common_times)
        for row in manifest:
            row['aligned_count'] = len(common_times)
            row['dropped_count'] = row['source_count']-len(common_times)
        return data, reference, common_times, manifest, handles
    except Exception:
        for ds in handles:
            ds.close()
        raise


def diagnostic_fingerprint(manifest, settings):
    text = json.dumps({'files':manifest,'settings':settings},sort_keys=True,default=str)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def save_diagnostic_nc(ds, path):
    """写入同目录临时文件后原子替换；不接触原始预测或模型文件。"""
    path = Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_name(path.stem+'.tmp-'+uuid.uuid4().hex+'.nc')
    encoding = {name:dict(zlib=True,complevel=3) for name,var in ds.data_vars.items()
                if var.dtype.kind in 'iuf' and var.ndim>0}
    try:
        ds.to_netcdf(temp,engine='netcdf4',encoding=encoding)
        os.replace(temp,path)
    finally:
        if temp.exists():
            temp.unlink()


def load_matching_cache(path, fingerprint):
    path = Path(path)
    if not path.exists():
        return None
    with xr.open_dataset(path,engine='netcdf4') as ds:
        if ds.attrs.get('input_fingerprint') != fingerprint:
            print('缓存与当前文件/配置不同，重新计算：',path.name)
            return None
        return ds.load()


def add_provenance(ds, fingerprint, manifest, experiments):
    ds = ds.copy(deep=False)
    ds.attrs['input_fingerprint'] = fingerprint
    ds.attrs['input_manifest_json'] = json.dumps(manifest,ensure_ascii=False)
    ds.attrs['experiment_metadata_json'] = json.dumps(experiments,ensure_ascii=False,default=str)
    ds.attrs['reference_pairing'] = 'Each experiment uses its own saved testy and predicty on common timestamps.'
    ds.attrs['spatial_derivatives'] = 'Training finite-difference operators; central secants inside and one-sided edges.'
    ds.attrs['scope'] = 'Diagnostics of fixed trained 1-hour models; no retraining or optimizer update.'
    if 'experiment' in ds.coords:
        label_map = {e['id']:e['label'] for e in experiments}
        ds = ds.assign_coords(experiment_label=('experiment',[label_map.get(str(e),str(e)) for e in ds.experiment.values]))
    return ds


for handle in globals().get('_diagnostic_handles', []):
    handle.close()
input_data, coordinates, evaluation_times, input_manifest, _diagnostic_handles = open_aligned_results(
    DATA_DIR, EXPERIMENTS, FILE_OVERRIDES, TIME_START, TIME_END)
latitude, longitude, levels = (coordinates[k] for k in ['latitude','longitude','level'])
settings = dict(version='physical-diagnostics-1.0',w_external_scale=W_EXTERNAL_SCALE,
                time_hash=hashlib.sha256(evaluation_times.tobytes()).hexdigest())
physics_fingerprint = diagnostic_fingerprint(input_manifest,settings)
display(pd.DataFrame(input_manifest)[['experiment','component','kind','source_count','aligned_count','dropped_count']])
print('共同测试时段：', evaluation_times[0], '至', evaluation_times[-1], '；时次数：',len(evaluation_times))
print('压力层：', levels, 'hPa；格点：',len(latitude),'×',len(longitude))


In [ ]:
import numpy as np
import xarray as xr


PHYSICS_METRICS = (
    'cont_pred_rms', 'cont_true_rms', 'cont_error_rmse',
    'vorticity_rmse', 'vorticity_r', 'vorticity_std_ratio',
    'shear_rmse', 'u_rmse', 'v_rmse', 'omega_rmse',
)
PHYSICS_STATISTICS = (
    'cont_pred_sq', 'cont_true_sq', 'cont_error_sq',
    'vort_error_sq', 'vort_pred_sum', 'vort_true_sum',
    'vort_pred_sq', 'vort_true_sq', 'vort_cross',
    'shear_error_sq', 'u_error_sq', 'v_error_sq', 'omega_error_sq',
)


def _physics_coordinate(values, name):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or len(values) < 2 or not np.all(np.isfinite(values)):
        raise ValueError(f'{name}: need at least two finite one-dimensional coordinates.')
    d = np.diff(values)
    if not (np.all(d > 0) or np.all(d < 0)):
        raise ValueError(f'{name}: coordinates must be strictly monotonic.')
    return values


def _secant_derivative(field, coord, axis):
    """Original PINN stencil: central secants, first-order one-sided edges."""
    a = np.moveaxis(np.asarray(field, dtype=np.float64), axis, -1)
    c = _physics_coordinate(coord, 'derivative coordinate')
    if a.shape[-1] != len(c):
        raise ValueError('Derivative coordinate length differs from field dimension.')
    out = np.empty_like(a)
    out[..., 0] = (a[..., 1] - a[..., 0]) / (c[1] - c[0])
    out[..., -1] = (a[..., -1] - a[..., -2]) / (c[-1] - c[-2])
    if len(c) > 2:
        out[..., 1:-1] = (a[..., 2:] - a[..., :-2]) / (c[2:] - c[:-2])
    return np.moveaxis(out, -1, axis)


def pressure_in_pa(levels):
    p = _physics_coordinate(levels, 'pressure levels')
    return p * 100.0 if np.max(np.abs(p)) < 2000.0 else p


def physics_fields(u, v, omega, lat, lon, levels):
    """Inputs are physical u/v [m s-1], omega [Pa s-1], shape (T,L,H,W)."""
    lat = _physics_coordinate(lat, 'latitude')
    lon = _physics_coordinate(lon, 'longitude')
    p = pressure_in_pa(levels)
    radius = 6371000.0
    y = radius * np.deg2rad(lat)
    coslat = np.cos(np.deg2rad(lat))
    coslat = np.where(np.abs(coslat) < 1e-8, 1e-8, coslat)
    denominator = radius * coslat[None, None, :, None]
    dxu = _secant_derivative(u, np.deg2rad(lon), -1) / denominator
    dxv = _secant_derivative(v, np.deg2rad(lon), -1) / denominator
    dyu = _secant_derivative(u, y, -2)
    dyv = _secant_derivative(v, y, -2)
    return (
        dxu + dyv + _secant_derivative(omega, p, 1),
        dxv - dyu,
        _secant_derivative(u, p, 1),
        _secant_derivative(v, p, 1),
    )


def _safe_ratio(a, b):
    a, b = np.broadcast_arrays(a, b)
    out = np.full(a.shape, np.nan, dtype=np.float64)
    np.divide(a, b, out=out, where=np.isfinite(b) & (b > 0))
    return out


def _metrics_from_statistics(sums, count):
    """The statistic axis is last; every other axis broadcasts with count."""
    d = {name: sums[..., i] for i, name in enumerate(PHYSICS_STATISTICS)}
    n = np.asarray(count, dtype=np.float64)
    mp = _safe_ratio(d['vort_pred_sum'], n)
    mt = _safe_ratio(d['vort_true_sum'], n)
    vp = np.maximum(_safe_ratio(d['vort_pred_sq'], n) - mp * mp, 0.0)
    vt = np.maximum(_safe_ratio(d['vort_true_sq'], n) - mt * mt, 0.0)
    cov = _safe_ratio(d['vort_cross'], n) - mp * mt
    tol = 64.0 * np.finfo(np.float64).eps
    vp = np.where(vp <= tol * np.maximum(_safe_ratio(d['vort_pred_sq'], n), mp * mp), 0.0, vp)
    vt = np.where(vt <= tol * np.maximum(_safe_ratio(d['vort_true_sq'], n), mt * mt), 0.0, vt)
    r = np.clip(_safe_ratio(cov, np.sqrt(vp * vt)), -1.0, 1.0)
    r = np.where(n >= 2, r, np.nan)
    ratio = np.sqrt(_safe_ratio(vp, vt))
    rm = lambda key: np.sqrt(np.maximum(_safe_ratio(d[key], n), 0.0))
    return np.stack([
        rm('cont_pred_sq'), rm('cont_true_sq'), rm('cont_error_sq'),
        rm('vort_error_sq'), r, ratio, rm('shear_error_sq'),
        rm('u_error_sq'), rm('v_error_sq'), rm('omega_error_sq'),
    ], axis=-1)


def _physics_terms(raw, derived, eid):
    up, vp, op = raw[eid]['predicty']
    ut, vt, ot = raw[eid]['testy']
    cp, zp, sup, svp = derived[eid]['predicty']
    ct, zt, sut, svt = derived[eid]['testy']
    yield cp * cp
    yield ct * ct
    yield (cp - ct) ** 2
    yield (zp - zt) ** 2
    yield zp
    yield zt
    yield zp * zp
    yield zt * zt
    yield zp * zt
    yield (sup - sut) ** 2 + (svp - svt) ** 2
    yield (up - ut) ** 2
    yield (vp - vt) ** 2
    yield (op - ot) ** 2


def _aggregation_names(levels):
    p = pressure_in_pa(levels) / 100.0
    return ['all_levels'] + [f'{v:g}hPa' for v in p]


def _summarize_physics(stats):
    s = stats['sum'].sum('time').values  # experiment,level,statistic
    n = stats['count'].sum('time').values
    a = np.concatenate([s.sum(axis=1, keepdims=True), s], axis=1)
    an = np.concatenate([[n.sum()], n])
    vals = _metrics_from_statistics(a, an[None, :])
    return xr.Dataset(
        {'value': (('experiment', 'aggregation', 'metric'), vals),
         'n_valid': (('aggregation',), an.astype(np.int64))},
        coords={'experiment': stats.experiment.values,
                'aggregation': _aggregation_names(stats.level.values),
                'metric': list(PHYSICS_METRICS)},
        attrs=dict(stats.attrs),
    )


def collect_physics(input_data, lat, lon, levels, w_external_scale=-0.1,
                    chunk_size=48, progress=True):
    """Stream aligned own-truth/prediction pairs; return summary,stats,maps.

    input_data[eid]['u'/'v'/'w']['testy'/'predicty'] must be aligned xarray
    DataArrays with dimensions time,level,latitude,longitude. All experiments,
    raw components and diagnostic fields share one finite-support mask. This
    gives identical spatiotemporal support and preserves own reference data.
    """
    ids = list(input_data)
    if not ids:
        raise ValueError('No experiment supplied.')
    if not np.isfinite(w_external_scale) or w_external_scale == 0:
        raise ValueError('w_external_scale must be finite and nonzero.')
    if int(chunk_size) != chunk_size or chunk_size < 1:
        raise ValueError('chunk_size must be a positive integer.')
    lat = _physics_coordinate(lat, 'latitude')
    lon = _physics_coordinate(lon, 'longitude')
    levels = _physics_coordinate(levels, 'levels')
    template = input_data[ids[0]]['u']['testy']
    time = template.time.values
    shape = (len(time), len(levels), len(lat), len(lon))
    dimensions = ('time', 'level', 'latitude', 'longitude')
    if len(time) < 2:
        raise ValueError('Need at least two aligned test times.')
    for eid in ids:
        for component in ('u', 'v', 'w'):
            for role in ('testy', 'predicty'):
                da = input_data[eid][component][role]
                if da.dims != dimensions or da.shape != shape:
                    raise ValueError(f'{eid}/{component}/{role}: wrong dimensions or shape.')
                for dim, expected in zip(dimensions, (time, levels, lat, lon)):
                    if not np.array_equal(da[dim].values, expected):
                        raise ValueError(f'{eid}/{component}/{role}: {dim} is not aligned.')
    ne, nt, nl, nh, nw = len(ids), *shape
    ns = len(PHYSICS_STATISTICS)
    time_sums = np.zeros((ne, nt, nl, ns), dtype=np.float64)
    time_count = np.zeros((nt, nl), dtype=np.int64)
    map_sums = np.zeros((ne, nl, nh, nw, ns), dtype=np.float64)
    map_count = np.zeros((nl, nh, nw), dtype=np.int64)
    for start in range(0, nt, int(chunk_size)):
        stop = min(start + int(chunk_size), nt)
        raw, derived = {}, {}
        common = np.ones((stop-start, nl, nh, nw), dtype=bool)
        for eid in ids:
            raw[eid], derived[eid] = {}, {}
            for role in ('testy', 'predicty'):
                fields = tuple(np.asarray(input_data[eid][c][role].isel(
                    time=slice(start, stop)).values, dtype=np.float64)
                    for c in ('u', 'v', 'w'))
                fields = (fields[0], fields[1], fields[2] / w_external_scale)
                raw[eid][role] = fields
                derived[eid][role] = physics_fields(*fields, lat, lon, levels)
                for field in fields + derived[eid][role]:
                    common &= np.isfinite(field)
        time_count[start:stop] = common.sum(axis=(-2, -1), dtype=np.int64)
        map_count += common.sum(axis=0, dtype=np.int64)
        for ie, eid in enumerate(ids):
            for j, term in enumerate(_physics_terms(raw, derived, eid)):
                term = np.where(common, term, 0.0)
                time_sums[ie, start:stop, :, j] = term.sum(axis=(-2, -1))
                map_sums[ie, ..., j] += term.sum(axis=0)
        if progress and (start == 0 or stop == nt or start // int(chunk_size) % 50 == 0):
            print(f'Physical diagnostics: {stop:,}/{nt:,} aligned times', flush=True)
    metadata = {
        'aggregation': 'Unweighted pooled space-time metrics; all_levels also pools levels. '
                       'These are not means of grid-point R/RMSPE from Fig. S2.',
        'finite_support': 'One common mask across all experiments, own truths/predictions, '
                          'raw u/v/omega and all continuity/vorticity/shear fields.',
        'truth_pairing': 'Every experiment uses its own testy and predicty.',
        'derivative_scheme': 'Original PINN central secants; one-sided domain/pressure edges; '
                             'local Cartesian spherical-coordinate spacings, R=6371000 m. '
                             'No spherical curvature terms are added.',
        'pressure_unit': 'Pa for differentiation; values<2000 converted from hPa.',
        'w_external_scale': float(w_external_scale),
        'omega_conversion': 'omega = stored_w / w_external_scale',
        'shear_definition': 'sqrt(mean[(dudp_pred-dudp_true)^2+(dvdp_pred-dvdp_true)^2])',
        'vorticity_std_ratio_definition': 'pooled population std(predicted vorticity) / '
                                          'pooled population std(own reference vorticity)',
        'continuity_reference': 'ERA5 own-reference residual retained; residual need not be zero '
                                'on three pressure levels and finite-difference grid.',
        'metric_units': 'continuity/vorticity RMS or RMSE: s-1; shear RMSE: m s-1 Pa-1; '
                        'u/v RMSE: m s-1; omega RMSE: Pa s-1; R/std_ratio: dimensionless',
        'n_aligned_times': int(nt),
    }
    coords = {'experiment': ids, 'time': time, 'level': levels,
              'statistic': list(PHYSICS_STATISTICS)}
    stats = xr.Dataset(
        {'sum': (('experiment', 'time', 'level', 'statistic'), time_sums),
         'count': (('time', 'level'), time_count)}, coords=coords, attrs=metadata)
    map_values = _metrics_from_statistics(map_sums, map_count[None, ...])
    maps = xr.Dataset(
        {m: (('experiment', 'level', 'latitude', 'longitude'), map_values[..., i])
         for i, m in enumerate(PHYSICS_METRICS)},
        coords={'experiment': ids, 'level': levels, 'latitude': lat, 'longitude': lon},
        attrs=dict(metadata))
    maps['n_valid'] = (('level', 'latitude', 'longitude'), map_count)
    maps.attrs['aggregation'] = 'At each level/grid point, metrics pool aligned times only.'
    return _summarize_physics(stats), stats, maps


def valid_hourly_block_starts(times, length):
    """Only blocks wholly inside uninterrupted, unique, increasing hourly runs."""
    t = np.asarray(times).astype('datetime64[ns]')
    if np.any(np.isnat(t)) or len(t) < 2 or np.any(np.diff(t) <= np.timedelta64(0, 'ns')):
        raise ValueError('Bootstrap times must be unique, finite and strictly increasing.')
    if int(length) != length or length < 1:
        raise ValueError('Block length must be a positive integer number of hours.')
    length = int(length)
    if length > len(t):
        return np.empty(0, dtype=np.int64)
    if length == 1:
        return np.arange(len(t), dtype=np.int64)
    gaps = np.diff(t) != np.timedelta64(1, 'h')
    prefix = np.concatenate(([0], np.cumsum(gaps, dtype=np.int64)))
    start = np.arange(len(t) - length + 1, dtype=np.int64)
    return start[(prefix[start + length - 1] - prefix[start]) == 0]


def bootstrap_physics(stats, block_hours=(72, 24, 168), n_boot=1000, seed=25,
                      baseline='no_pinn', keep_draws=True, progress=True):
    """Paired non-circular hourly moving-block bootstrap on additive moments.

    Each replicate draws full blocks uniformly with replacement from every
    valid block start. The final independently sampled full block is truncated
    so exactly N aligned times are resampled. Blocks never span a time gap.
    The same draws are used for all experiments, levels, variables and metrics.
    CIs describe temporal test-sample uncertainty conditional on trained models;
    they are neither training-seed uncertainty nor post-selection-adjusted CIs.
    """
    ids = list(map(str, stats.experiment.values))
    if baseline not in ids:
        raise ValueError(f'Baseline {baseline!r} is missing.')
    if int(n_boot) != n_boot or n_boot < 2:
        raise ValueError('Need at least two bootstrap replicates.')
    lengths = list(dict.fromkeys(int(b) for b in block_hours))
    if not lengths or any(b < 1 for b in lengths):
        raise ValueError('At least one positive block length is required.')
    a = np.asarray(stats['sum'].values, dtype=np.float64)  # E,T,L,S
    n = np.asarray(stats['count'].values, dtype=np.float64)
    ne, nt, nl, ns = a.shape
    if ns != len(PHYSICS_STATISTICS) or list(map(str, stats.statistic.values)) != list(PHYSICS_STATISTICS):
        raise ValueError('Sufficient-statistic names/order differ from this implementation.')
    prefix = np.concatenate([np.zeros((ne, 1, nl, ns)), np.cumsum(a, axis=1)], axis=1)
    prefix_n = np.concatenate([np.zeros((1, nl)), np.cumsum(n, axis=0)], axis=0)
    draws = np.full((len(lengths), int(n_boot), ne, nl+1, len(PHYSICS_METRICS)), np.nan)
    counts = []
    for il, length in enumerate(lengths):
        starts = valid_hourly_block_starts(stats.time.values, length)
        if len(starts) == 0:
            raise ValueError(f'No uninterrupted hourly block of {length} h. '
                             'Choose a shorter block length; gaps are never bridged.')
        coverage = np.zeros(nt+1, dtype=np.int64)
        np.add.at(coverage, starts, 1)
        np.add.at(coverage, starts+length, -1)
        uncovered = np.cumsum(coverage[:-1]) == 0
        if np.any(uncovered):
            raise ValueError(f'{int(uncovered.sum())} aligned times lie in hourly runs shorter '
                             f'than {length} h. Choose a shorter block or explicitly restrict '
                             'the evaluation interval; no test times are silently discarded.')
        counts.append(len(starts))
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), length]))
        nblocks, rem = divmod(nt, length)
        for ib in range(int(n_boot)):
            chosen = rng.choice(starts, size=nblocks+(rem > 0), replace=True)
            lengths_drawn = np.full(len(chosen), length, dtype=np.int64)
            if rem:
                lengths_drawn[-1] = rem
            stops = chosen + lengths_drawn
            total = (prefix[:, stops] - prefix[:, chosen]).sum(axis=1)
            total_n = (prefix_n[stops] - prefix_n[chosen]).sum(axis=0)
            agg = np.concatenate([total.sum(axis=1, keepdims=True), total], axis=1)
            agg_n = np.concatenate([[total_n.sum()], total_n])
            draws[il, ib] = _metrics_from_statistics(agg, agg_n[None, :])
        if progress:
            print(f'Bootstrap: {length} h, {int(n_boot):,} paired draws, '
                  f'{len(starts):,} eligible starts', flush=True)
    estimate = _summarize_physics(stats)['value'].values
    bi = ids.index(baseline)
    delta = estimate - estimate[bi:bi+1]
    relative = 100.0 * _safe_ratio(estimate[bi:bi+1]-estimate, estimate[bi:bi+1])
    delta_draw = draws - draws[:, :, bi:bi+1]
    relative_draw = 100.0 * _safe_ratio(draws[:, :, bi:bi+1]-draws, draws[:, :, bi:bi+1])
    not_error = [PHYSICS_METRICS.index(m) for m in ('vorticity_r', 'vorticity_std_ratio')]
    relative[..., not_error] = np.nan
    relative_draw[..., not_error] = np.nan

    def interval(x):
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            return np.nanpercentile(x, [2.5, 97.5], axis=1)

    raw_ci = interval(draws)
    delta_ci = interval(delta_draw)
    relative_ci = interval(relative_draw)
    edims = ('experiment', 'aggregation', 'metric')
    cdims = ('block_hours',) + edims
    ds = xr.Dataset({
        'estimate': (edims, estimate),
        'ci_low': (cdims, raw_ci[0]), 'ci_high': (cdims, raw_ci[1]),
        'delta': (edims, delta),
        'delta_ci_low': (cdims, delta_ci[0]), 'delta_ci_high': (cdims, delta_ci[1]),
        'relative_reduction_pct': (edims, relative),
        'relative_reduction_ci_low': (cdims, relative_ci[0]),
        'relative_reduction_ci_high': (cdims, relative_ci[1]),
        'n_finite_draws': (cdims, np.isfinite(draws).sum(axis=1).astype(np.int32)),
        'n_finite_relative_draws': (cdims, np.isfinite(relative_draw).sum(axis=1).astype(np.int32)),
        'n_eligible_block_starts': (('block_hours',), np.asarray(counts, dtype=np.int64)),
    }, coords={'experiment': ids, 'aggregation': _aggregation_names(stats.level.values),
               'metric': list(PHYSICS_METRICS), 'block_hours': lengths},
       attrs=dict(stats.attrs))
    if keep_draws:
        ds = ds.assign_coords(replicate=np.arange(int(n_boot), dtype=np.int32))
        ddims = ('block_hours', 'replicate') + edims
        ds['bootstrap_value'] = (ddims, draws)
        ds['bootstrap_delta'] = (ddims, delta_draw)
        ds['bootstrap_relative_reduction_pct'] = (ddims, relative_draw)
    ds.attrs.update({
        'baseline': baseline, 'bootstrap_seed': int(seed), 'bootstrap_replicates': int(n_boot),
        'confidence_level': 0.95, 'primary_block_hours': int(lengths[0]),
        'bootstrap_method': 'Paired non-circular moving-block percentile bootstrap; '
                            'valid consecutive hourly blocks only, full spatial field retained, '
                            'exactly N times per replicate; final sampled full block truncated.',
        'difference_direction': 'delta = experiment minus baseline; '
                                'relative_reduction_pct = 100*(baseline-experiment)/baseline.',
        'relative_reduction_caution': 'Positive residual/error reduction is lower magnitude; '
                                    'not an unconditional statement of physical realism. '
                                    'Undefined at zero baseline. Not reported for R/std-ratio.',
        'uncertainty_scope': 'Conditional temporal test-sample uncertainty for the saved fitted '
                             'models; not seed uncertainty, not adjusted for configuration '
                             'selection or multiple comparisons. Block sensitivity is reported.',
        'gap_handling': 'No block spans a gap; fail if any time belongs only to a run shorter '
                        'than the requested block length. No silent time exclusion.',
    })
    return ds


In [ ]:
physical_summary = load_matching_cache(SUMMARY_NC,physics_fingerprint) if REUSE_CACHE else None
physical_stats = load_matching_cache(STATS_NC,physics_fingerprint) if REUSE_CACHE else None
physical_maps = load_matching_cache(MAPS_NC,physics_fingerprint) if REUSE_CACHE else None
if any(ds is None for ds in [physical_summary,physical_stats,physical_maps]):
    physical_summary,physical_stats,physical_maps = collect_physics(
        input_data,latitude,longitude,levels,w_external_scale=W_EXTERNAL_SCALE,chunk_size=TIME_CHUNK)
    physical_summary = add_provenance(physical_summary,physics_fingerprint,input_manifest,EXPERIMENTS)
    physical_stats = add_provenance(physical_stats,physics_fingerprint,input_manifest,EXPERIMENTS)
    physical_maps = add_provenance(physical_maps,physics_fingerprint,input_manifest,EXPERIMENTS)
    for ds,path in [(physical_summary,SUMMARY_NC),(physical_stats,STATS_NC),(physical_maps,MAPS_NC)]:
        save_diagnostic_nc(ds,path)
else:
    print('已读取匹配的物理指标缓存。')


In [ ]:
bootstrap_fingerprint = diagnostic_fingerprint(input_manifest,
    dict(physics=physics_fingerprint,block_hours=list(BLOCK_HOURS),n_boot=N_BOOTSTRAP,seed=BOOTSTRAP_SEED))
bootstrap_results = load_matching_cache(BOOTSTRAP_NC,bootstrap_fingerprint) if REUSE_CACHE else None
if bootstrap_results is None:
    bootstrap_results = bootstrap_physics(physical_stats,block_hours=BLOCK_HOURS,
        n_boot=N_BOOTSTRAP,seed=BOOTSTRAP_SEED,baseline='no_pinn',keep_draws=True)
    bootstrap_results = add_provenance(bootstrap_results,bootstrap_fingerprint,input_manifest,EXPERIMENTS)
    save_diagnostic_nc(bootstrap_results,BOOTSTRAP_NC)
else:
    print('已读取匹配的分块区间缓存。')


In [ ]:
def build_original_diagnostic_model(height, width):
    from direct11_model import build_model
    from pinn_loss import PINNLoss
    if (height, width) != (49, 73):
        raise ValueError('The study network expects the native 49 by 73 grid.')
    return build_model(history_steps=6, input_channels=3), PINNLoss


def diagnostic_loss_terms(loss, y_pred, y_true):
    data, continuity, vorticity, shear = loss.component_losses(y_pred, y_true)
    terms = {'data': data, 'continuity': loss.lambda_continuity * continuity,
             'vorticity': loss.lambda_vorticity * vorticity, 'shear': loss.lambda_shear * shear}
    terms['physics'] = terms['continuity'] + terms['vorticity'] + terms['shear']
    return terms

def parameter_gradient_summary(model, terms):
    """Euclidean norms/cosines over ALL trainable model parameters, not outputs."""
    import numpy as np
    import torch
    params = tuple(p for p in model.parameters() if p.requires_grad)
    grads, norms = {}, {}
    keys = tuple(terms)
    for i, key in enumerate(keys):
        gs = torch.autograd.grad(terms[key], params, retain_graph=(i < len(keys)-1),
                                 create_graph=False, allow_unused=True)
        grads[key] = tuple(None if v is None else v.detach() for v in gs)
        norms[key] = float(sum(float((v.double() ** 2).sum().item())
                              for v in grads[key] if v is not None) ** 0.5)
    norm_data = norms['data']
    result = {'gradient_norm': norms, 'gradient_ratio_to_data': {}, 'cosine_to_data': {}}
    for key in keys:
        result['gradient_ratio_to_data'][key] = norms[key] / norm_data if norm_data > 0 else np.nan
        dot = sum(float((a.double() * b.double()).sum().item())
                  for a, b in zip(grads[key], grads['data']) if a is not None and b is not None)
        result['cosine_to_data'][key] = float(np.clip(dot / (norms[key]*norm_data), -1., 1.)) if norms[key] > 0 and norm_data > 0 else np.nan
    return result


def reproduction_error(actual, expected, channel_std):
    """RMS of prediction mismatch normalized by ORIGINAL per-grid target std."""
    import numpy as np
    actual, expected = np.asarray(actual), np.asarray(expected)
    if actual.shape != expected.shape or not (np.isfinite(actual).all() and np.isfinite(expected).all()):
        return np.inf
    scale = np.asarray(channel_std, dtype=np.float64)
    if not np.isfinite(scale).all() or np.any(scale <= 0):
        return np.inf
    return float(np.sqrt(np.mean(((actual.astype(np.float64)-expected)/scale) ** 2)))


def _gradient_sample_times(evaluation_times, samples_per_season=2):
    """Deterministic equally spaced picks within each meteorological season."""
    import numpy as np
    ts = np.unique(np.asarray(evaluation_times, dtype='datetime64[ns]'))
    month = ts.astype('datetime64[M]').astype(int) % 12 + 1
    selected = []
    for months in ((12, 1, 2), (3, 4, 5), (6, 7, 8), (9, 10, 11)):
        candidates = ts[np.isin(month, months)]
        if len(candidates):
            ix = np.floor((np.arange(min(samples_per_season, len(candidates))) + .5)
                          * len(candidates) / min(samples_per_season, len(candidates))).astype(int)
            selected.extend(candidates[ix])
    return np.unique(np.asarray(selected, dtype='datetime64[ns]'))


def _gradient_raw_specs(raw_dir):
    from pathlib import Path
    p = Path(raw_dir)
    specs = [(p/'Surface-pressure.nc', 'sp', 1.0, 'x'),
             (p/'2m-temperature.nc', 't2m', 1.0, 'x'),
             (p/'2m-dewpoint-temperature.nc', 'd2m', 1.0, 'x')]
    for prefix, var, scale in [('U-component-of-wind', 'u', 1.0),
                               ('V-component-of-wind', 'v', 1.0),
                               ('Vertical-velocity', 'w', -0.1)]:
        specs.extend((p/f'{prefix}-{lev}hpa.nc', var, scale, 'y') for lev in (850,925,1000))
    return specs


def _gradient_open_raw(path, variable, start, end, latitude=None, longitude=None):
    import numpy as np
    import xarray as xr
    ds = xr.open_dataset(path)
    try:
        da = ds[variable]
        rename = {k: v for k, v in {'valid_time':'time','lat':'latitude','lon':'longitude'}.items()
                  if k in da.dims and v not in da.dims}
        da = da.rename(rename)
        for dim in tuple(da.dims):
            if dim not in ('time','latitude','longitude'):
                if da.sizes[dim] != 1:
                    raise ValueError(f'{path}: unexpected non-singleton dimension {dim}; select it explicitly.')
                da = da.isel({dim: 0}, drop=True)
        da = da.transpose('time','latitude','longitude')

        time_original = da.time.values.astype('datetime64[ns]')
        time_rounded = (
            time_original + np.timedelta64(30, 'm')
        ).astype('datetime64[h]').astype('datetime64[ns]')

        roundoff_only = (
            ~np.isnat(time_original)
        ) & (
            np.abs(time_rounded - time_original) <= np.timedelta64(1, 'us')
        )

        time_corrected = np.where(
            roundoff_only, time_rounded, time_original
        )

        da = da.assign_coords(time=time_corrected).sel(
            time=slice(start, end)
        )
        if latitude is None:
            da = da.isel(latitude=np.flatnonzero((da.latitude.values >= 24) & (da.latitude.values <= 36)))
        else:
            da = da.sel(latitude=np.asarray(latitude))
        if longitude is None:
            da = da.isel(longitude=np.flatnonzero((da.longitude.values >= 108) & (da.longitude.values <= 126)))
        else:
            da = da.sel(longitude=np.asarray(longitude))
        ts = da.time.values.astype('datetime64[ns]')
        if len(ts) < 7 or not np.all(np.diff(ts) == np.timedelta64(1,'h')):
            raise ValueError(f'{path}: original data must have unique, consecutive hourly times.')
        if ts[0] != np.datetime64(start,'ns') or ts[-1] != np.datetime64(end,'ns'):
            raise ValueError(f'{path}: full normalization period endpoints are missing.')
        return ds, da
    except Exception:
        ds.close()
        raise


def _gradient_prepare_stats(config, latitude, longitude):
    path = Path(config['normalization_cache'])
    stats = {}
    with np.load(path, allow_pickle=False) as saved:
        for role, channels in [('x', 3), ('y', 9)]:
            for kind in ('mean', 'std'):
                key = f'{role}_{kind}'
                value = np.asarray(saved[f'v{key}'])
                if value.shape != (len(latitude), len(longitude), channels):
                    raise ValueError(f'{path}: unexpected shape of v{key}: {value.shape}')
                stats[key] = value.transpose(2, 0, 1)
    return stats, json.dumps({'source': str(path), 'fit': 'training split'}, ensure_ascii=False)
def run_gradient_diagnostic(gradient_config, experiments, evaluation_times,
                            latitude, longitude, levels, read_pair):
    """Return auditable per-sample diagnostics; unavailable/unverified models stay NaN.

    read_pair(experiment_id, component, sample_times) -> own (testy, predicty),
    shape (time,level,latitude,longitude), using caller's coordinate order.
    Each experiments value must contain weights=(lambda_c,lambda_v,lambda_s), model_path.
    """
    import gc
    import hashlib
    import json
    import numpy as np
    import xarray as xr
    from pathlib import Path
    cfg = dict(samples_per_season=2, prediction_rrmse_tolerance=1e-3,
               truth_rrmse_tolerance=1e-3, experiment_ids=['full_pinn', 'no_shear'])
    cfg.update(gradient_config)
    ids = [eid for eid in cfg['experiment_ids'] if eid in experiments]
    ts = _gradient_sample_times(evaluation_times,int(cfg['samples_per_season']))
    terms = ['data','continuity','vorticity','shear','physics']
    coords = dict(experiment=ids,sample=np.arange(len(ts)),term=terms,component=['u','v','w'],
                  sample_time=('sample',ts))
    ds = xr.Dataset(coords=coords)
    for name in ('loss','loss_ratio_to_data','gradient_norm','gradient_ratio_to_data','cosine_to_data'):
        ds[name] = (('experiment','sample','term'),np.full((len(ids),len(ts),len(terms)),np.nan))
    for name in ('prediction_reproduction_rrmse','truth_reproduction_rrmse'):
        ds[name] = (('experiment','sample','component'),np.full((len(ids),len(ts),3),np.nan))
    ds['trusted_sample'] = (('experiment','sample'),np.zeros((len(ids),len(ts)),np.int8))
    ds['weight'] = (('experiment','term'),np.array([[1.,*experiments[eid]['weights'],np.nan] for eid in ids]).reshape(len(ids),5))
    status = ['not_evaluated'] * len(ids)
    reasons = [''] * len(ids)
    hashes = [''] * len(ids)
    ds.attrs.update(normalization='Training input and target scalers from the one-hour run.',
                    supervised_reference='Saved test labels paired with each checkpoint.',
                    diagnostic_config_json=json.dumps(cfg,default=str),
                    reproduction_metric='RMS mismatch divided by the training target standard deviation.')
    def finish():
        ds['status'] = ('experiment',np.asarray(status,dtype=str))
        ds['status_reason'] = ('experiment',np.asarray(reasons,dtype=str))
        ds['checkpoint_sha256'] = ('experiment',np.asarray(hashes,dtype=str))
        return ds
    if not ids or not len(ts):
        return finish()
    available = []
    for i,eid in enumerate(ids):
        path = Path(experiments[eid]['model_path'])
        if path.is_file(): available.append(i)
        else: status[i],reasons[i] = 'unavailable',f'Missing checkpoint: {path}'
    if not available:
        print('Gradient diagnostic unavailable: saved model state_dict files were not found.')
        return finish()
    try:
        import torch
    except ImportError as exc:
        for i in available: status[i],reasons[i] = 'unavailable',str(exc)
        print('Gradient diagnostic unavailable: PyTorch is not installed; S3/S4 remain valid.')
        return finish()
    try:
        raw_specs = _gradient_raw_specs(cfg['raw_dir'])
        missing = [str(s[0]) for s in raw_specs if not s[0].is_file()]
        if missing: raise FileNotFoundError('Missing original raw input/target files: '+'; '.join(missing))
        source,first = _gradient_open_raw(*raw_specs[0][:2],cfg['period_start'],cfg['period_end'])
        try:
            lat_raw,lon_raw = first.latitude.values.copy(),first.longitude.values.copy()
        finally: source.close()
        def indexes(caller, wanted):
            caller = np.asarray(caller)
            out=[]
            for value in wanted:
                where=np.flatnonzero(np.isclose(caller,value,rtol=0,atol=1e-6))
                if len(where)!=1: raise ValueError(f'Coordinate {value} is absent or duplicated in saved results.')
                out.append(where[0])
            if len(out)!=len(caller): raise ValueError('Raw and saved prediction spatial domains differ.')
            return np.asarray(out)
        lati,loni = indexes(latitude,lat_raw),indexes(longitude,lon_raw)
        levi = indexes(np.asarray(levels,dtype=float)/100. if np.max(levels)>2000 else levels,[850.,925.,1000.])
        stats,stats_metadata = _gradient_prepare_stats(cfg,lat_raw,lon_raw)
        ds.attrs['normalization_provenance_json'] = stats_metadata
        history = ts[:,None] - np.arange(6,0,-1)[None,:]*np.timedelta64(1,'h')
        inputs,raw_targets = [],[]
        for path,var,scale,role in raw_specs:
            source,field = _gradient_open_raw(path,var,cfg['period_start'],cfg['period_end'],lat_raw,lon_raw)
            try:
                select=history.ravel() if role=='x' else ts
                values=np.asarray(field.sel(time=select).values * scale,dtype=np.float32)
                (inputs if role=='x' else raw_targets).append(values)
            finally: source.close()
        x=np.stack(inputs,axis=1).reshape(len(ts),6,3,len(lat_raw),len(lon_raw))
        x=(x-stats['x_mean'][None,None])/stats['x_std'][None,None]
        raw_targets=np.stack(raw_targets,axis=1)
        if not np.isfinite(x).all(): raise ValueError('Selected standardized model inputs contain nonfinite values.')
    except Exception as exc:
        for i in available: status[i],reasons[i]='unavailable',f'{type(exc).__name__}: {exc}'
        print('Gradient preparation unavailable:',exc)
        return finish()
    requested=str(cfg['device'])
    device=torch.device(requested if not requested.startswith('cuda') or torch.cuda.is_available() else 'cpu')
    ds.attrs.update(device=str(device),torch_version=torch.__version__,samples_per_forward=1)
    print('Gradient diagnostic device:',device,'; samples:',len(ts))
    for i in available:
        eid=ids[i]
        model=None
        try:
            print('Gradient diagnostic:',eid)
            path=Path(experiments[eid]['model_path'])
            digest=hashlib.sha256()
            with path.open('rb') as f:
                for block in iter(lambda:f.read(1024*1024),b''): digest.update(block)
            hashes[i]=digest.hexdigest()
            model,loss_class=build_original_diagnostic_model(len(lat_raw),len(lon_raw))
            state=torch.load(path,map_location='cpu',weights_only=True)
            if isinstance(state,dict) and 'state_dict' in state: state=state['state_dict']
            model.load_state_dict(state,strict=True)
            model.to(device).eval()
            own_truth,own_prediction=[],[]
            for component in ('u','v','w'):
                truth,prediction=read_pair(eid,component,ts)
                own_truth.append(np.asarray(truth)[:,levi][:,:,lati][:,:,:,loni])
                own_prediction.append(np.asarray(prediction)[:,levi][:,:,lati][:,:,:,loni])
            own_truth=np.concatenate(own_truth,axis=1)
            own_prediction=np.concatenate(own_prediction,axis=1)
            reproduced=[]
            with torch.no_grad():
                for j in range(len(ts)):
                    yp=model(torch.as_tensor(x[j:j+1],dtype=torch.float32,device=device)).cpu().numpy()
                    reproduced.append(yp.astype(np.float64)*stats['y_std'][None]+stats['y_mean'][None])
            reproduced=np.concatenate(reproduced,axis=0)
            for j in range(len(ts)):
                for k in range(3):
                    part=slice(k*3,(k+1)*3)
                    ds['prediction_reproduction_rrmse'].values[i,j,k]=reproduction_error(reproduced[j,part],own_prediction[j,part],stats['y_std'][part])
                    ds['truth_reproduction_rrmse'].values[i,j,k]=reproduction_error(raw_targets[j,part],own_truth[j,part],stats['y_std'][part])
            good=(np.all(ds.prediction_reproduction_rrmse.values[i]<=cfg['prediction_rrmse_tolerance'])
                  and np.all(ds.truth_reproduction_rrmse.values[i]<=cfg['truth_rrmse_tolerance']))
            if not good:
                status[i]='reproduction_failed'
                reasons[i]=f'Checkpoint/raw input reproduction failed; maximum prediction standardized RMS mismatch={np.nanmax(ds.prediction_reproduction_rrmse.values[i]):.6g}, truth mismatch={np.nanmax(ds.truth_reproduction_rrmse.values[i]):.6g}. No gradients reported.'
                print(eid+': '+reasons[i])
                continue
            loss=loss_class(stats['y_mean'],stats['y_std'],*experiments[eid]['weights'],
                            lat=lat_raw,lon=lon_raw,pressure_levels=[850.,925.,1000.],w_external_scale=-.1)
            standardized_truth=np.asarray((own_truth-stats['y_mean'][None])/stats['y_std'][None],dtype=np.float32)
            for j in range(len(ts)):
                pred=model(torch.as_tensor(x[j:j+1],dtype=torch.float32,device=device))
                target=torch.as_tensor(standardized_truth[j:j+1],dtype=torch.float32,device=device)
                values=diagnostic_loss_terms(loss,pred,target)
                summary=parameter_gradient_summary(model,values)
                data_value=float(values['data'].detach().cpu())
                for k,term in enumerate(terms):
                    value=float(values[term].detach().cpu())
                    ds['loss'].values[i,j,k]=value
                    ds['loss_ratio_to_data'].values[i,j,k]=value/data_value if data_value>0 else np.nan
                    for key in ('gradient_norm','gradient_ratio_to_data','cosine_to_data'):
                        ds[key].values[i,j,k]=summary[key][term]
                ds['trusted_sample'].values[i,j]=1
                print(f'  {j+1}/{len(ts)} {str(ts[j])[:16]}: physics/data loss={ds.loss_ratio_to_data.values[i,j,-1]:.3g}; gradient={ds.gradient_ratio_to_data.values[i,j,-1]:.3g}')
            status[i]='verified'
            reasons[i]='Strict state_dict load and per-component prediction/truth reproduction checks passed.'
        except Exception as exc:
            status[i]='failed'
            reasons[i]=f'{type(exc).__name__}: {exc}'
            ds['trusted_sample'].values[i]=0
            for key in ('loss','loss_ratio_to_data','gradient_norm','gradient_ratio_to_data','cosine_to_data'):
                ds[key].values[i]=np.nan
            print(eid+': gradient diagnostic failed:',exc)
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    return finish()


In [ ]:
GRADIENT_CONFIG['prediction_rrmse_tolerance'] = 2.e-3
gradient_results = None
gradient_plot_data = pd.DataFrame()
if RUN_GRADIENT:
    def read_saved_pair(eid,component,times):
        return tuple(np.asarray(input_data[eid][component][kind].sel(time=times).values)
                     for kind in ['testy','predicty'])
    gradient_results = run_gradient_diagnostic(
        GRADIENT_CONFIG,{e['id']:e for e in EXPERIMENTS},evaluation_times,
        latitude,longitude,levels,read_saved_pair)
    gradient_results = add_provenance(gradient_results,physics_fingerprint,input_manifest,EXPERIMENTS)
    save_diagnostic_nc(gradient_results,GRADIENT_NC)

    failed = gradient_results.experiment.values[gradient_results.status.values != 'verified']
    if len(failed):
        details = [f'{exp}: {gradient_results.status_reason.sel(experiment=exp).item()}' for exp in failed]
        raise ValueError('Gradient diagnostics unavailable: ' + ' | '.join(details))
